# Nested Learning for CAD — HOPE-CAD v5 normal pilot

**Revision mới, chưa có benchmark gain.** Đọc `architecture_decision_v5.md` và `v5_implementation.md` trong `docs/main_spec/`.

Ba inputs: source `hope_cad_support_v5_source_v1.zip`, MVTec AD, ViT-B/8 `vit_base_patch8_224.augreg_in21k.safetensors`. Không cần W6/recovery/W7 evidence hoặc learned weights.

GPU T4 x2; **Save Version → Save & Run All**. Default `PHASE='pilot'`: preflight 24 real normals trên discarded clone, rồi bốn fresh jobs CADIC/GEOMETRIC/FIRST_ORDER/NESTED, mỗi job 256 normal images. Source chỉ extract train/good, không test/GT.

`SESSION_HOURS=3.0` là soft limit để pause/export, không phải ETA. Có heartbeat 30 giây. Resume bằng **latest v5 pilot output** cùng ba inputs, tháo outputs cũ hơn; source/protocol/runtime phải cùng identity. Chỉ muốn preflight riêng: đặt `PHASE='preflight'`, sau PASS attach output đó rồi chạy pilot.

`pilot_complete_pending_review` cần review artifacts. `pilot_complete_scientific_gate_failed` nghĩa chạy xong nhưng mechanism/risk gate không đạt: giữ ZIP/log, không retune hoặc launch anomaly benchmark. Pause không là completion. Hard platform kill có thể ngăn export.


In [ ]:
PHASE='pilot'  # pilot: preflight tích hợp + 4 jobs x256 normals; preflight: chỉ kiểm trước
SESSION_HOURS=3.0  # soft stop; lưu/resume ở atomic boundary; không phải ETA
SOURCE_OVERRIDE=None
RESUME_OVERRIDE=None   # chỉ latest support-v5 pilot output; lần đầu để None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

from pathlib import Path,PurePosixPath
import hashlib,json,zipfile,os,sys,time,shutil
STARTED=time.monotonic()
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_support_v5_source')
print('[BOOTSTRAP] Tìm source HOPE-CAD v5',flush=True)
def hash_file(p):
    with Path(p).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
def source_manifest(p):
    if p.is_dir():return json.loads((p/'bundle_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:
        if len(z.namelist())!=len(set(z.namelist())):raise ValueError('Duplicate source ZIP paths')
        m=json.loads(z.read('bundle_manifest.json'))
        if set(z.namelist())!=set(m['files'])|{'bundle_manifest.json'}:raise ValueError('Source inventory mismatch')
        return m
def target(name):
    rel=PurePosixPath(name)
    if rel.is_absolute() or '..' in rel.parts or rel.as_posix()!=name or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    p=ROOT/name
    if not p.resolve().is_relative_to(ROOT.resolve()):raise ValueError('Escaping source path')
    return p
if SOURCE_OVERRIDE:choices=[Path(SOURCE_OVERRIDE)]
else:
    choices=[]
    for p in INPUT.rglob('bundle_manifest.json'):
        if json.loads(p.read_text()).get('schema')=='hope_support_v5_source_v1':choices.append(p.parent)
    if not choices:
        for p in INPUT.rglob('*.zip'):
            try:
                if source_manifest(p).get('schema')=='hope_support_v5_source_v1':choices.append(p)
            except (KeyError,ValueError,zipfile.BadZipFile):pass
if len(choices)!=1:raise RuntimeError(f'Expected ONE support-v5 source, found {choices}; set SOURCE_OVERRIDE')
source=choices[0];manifest=source_manifest(source)
if manifest['schema']!='hope_support_v5_source_v1':raise ValueError('Wrong source schema')
ROOT.mkdir(parents=True,exist_ok=True)
if source.is_dir():
    for name,expected in manifest['files'].items():
        original=source/name;dest=target(name)
        if not original.is_file() or original.is_symlink() or hash_file(original)!=expected:raise ValueError('Source checksum mismatch: '+name)
        dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(original,dest)
else:
    with zipfile.ZipFile(source) as z:
        for name,expected in manifest['files'].items():
            dest=target(name);dest.parent.mkdir(parents=True,exist_ok=True)
            with z.open(name) as reader,dest.open('wb') as writer:shutil.copyfileobj(reader,writer)
            if hash_file(dest)!=expected:raise ValueError('Source checksum mismatch: '+name)
(ROOT/'bundle_manifest.json').write_text(json.dumps(manifest,sort_keys=True))
os.chdir(ROOT);sys.path.insert(0,str(ROOT));sys.dont_write_bytecode=True
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('[BOOTSTRAP] Verified source files:',len(manifest['files']),manifest['origin'],flush=True)
WORK=Path('/kaggle/working/hope_support_v5_pilot')
ASSETS=Path('/tmp/hope_support_v5_assets')
OUTPUT=Path('/kaggle/working/hope_cad_support_v5_pilot_evidence.zip')


In [ ]:
from notebook_support.support_v5.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,session_hours=SESSION_HOURS,started=STARTED,
    inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),
    RESUME_OVERRIDE=RESUME_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
